# Responses API: Scenario 16 Quote-To-Cash (Handoff)

This notebook mirrors `release_room.scenarios.scenario_16_quote_to_cash_handoff`. It teaches the `scenario-16-quote-to-cash-handoff` variant of the quote-to-cash family using the Microsoft Agent Framework **Handoff** orchestration pattern, grounded by the local `quote_to_cash_context` MCP server (no network, credentials, or setup).

## 1. Notebook Setup And Local Imports

Run from the project virtual environment after `python -m pip install -e .`.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start):
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "release_room").exists():
            return candidate
        nested = candidate / "responses-api-release-room"
        if (nested / "src" / "release_room").exists():
            return nested
    raise RuntimeError("Could not find responses-api-release-room project root.")

PROJECT_ROOT = find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

PROJECT_ROOT

## 2. From RPA Flow To Agentic Orchestration

The source screenshot shows a classic quote-to-cash automation: rigid, scripted RPA/API steps that move a quote request from CRM, through customer enrichment, product/SKU selection, and pricing/legal, to a final quote. The agentic version keeps the same business story but changes *how* the work is coordinated:

- **Goal-oriented agents with an LLM** replace hard-coded scripts. Each agent decides how to use its tools to meet its goal.
- **Orchestration-managed state** carries context between stages (customer context, then product context) instead of brittle variable passing.
- **Tool use via MCP** grounds each agent in real enterprise systems (CRM, catalog, pricing, legal).
- **Transient agent waves**: agents are spun up for a stage and deallocated once their context is stored, so only the agents that are needed exist at any moment.

The six roles are consistent across all five pattern variants: `QuoteTriggerAgent`, `CustomerContextAgent`, `SkuDiscoveryAgent`, `ProductFitAgent`, `PricingTermsAgent`, and `QuoteGenerationAgent`.

**How Handoff maps to the screenshot:** Where the screenshot hard-codes the order of waves, Handoff lets the start agent decide which specialist to route to next, so unnecessary stages can be skipped and the path adapts to the deal.

## 3. API Fit: Responses API

This project hosts the **Responses API**. The `/responses` request shape is stable and OpenAI-compatible, and the scenario is chosen **at server startup** with `--scenario`. The quote-to-cash orchestration and its MCP tools run server-side and never change the request/response contract.

Use the Responses API when the caller is a chat-style client that sends `input`/`stream` and expects a conversational reply. Use the Invocations API project instead when a system needs to pick the scenario per request and receive an application-owned JSON job result.

## 4. Scenario Metadata And Sample Payload

The scenario module defines a single `SCENARIO`. The sample `/responses` payload lives under `samples/`.

In [ ]:
from release_room.scenarios.scenario_16_quote_to_cash_handoff import SCENARIO
from release_room.scenarios import SCENARIOS_BY_ID, get_scenario
from release_room.notebook_helpers import (
    agent_roster,
    default_ollama_kwargs,
    mcp_tool_context,
    pattern_anatomy,
    responses_api_reference,
    scenario_summary,
    workflow_result_to_text,
)

scenario = SCENARIO
assert SCENARIOS_BY_ID["scenario-16-quote-to-cash-handoff"] is scenario
assert get_scenario("scenario-16-quote-to-cash-handoff") is scenario
scenario_summary(scenario)

In [ ]:
import json
sample_payload = json.loads((PROJECT_ROOT / "samples" / "scenario-16-quote-to-cash-handoff.json").read_text())
sample_payload

## 5. Microsoft Pattern Anatomy: Handoff

This variant uses the Microsoft Agent Framework **Handoff** orchestration pattern. The helper returns the generic anatomy for the pattern; the quote-to-cash specifics are layered on top by the agents and tools.

In [ ]:
pattern_anatomy(scenario)

## 6. Quote-To-Cash Flow Diagram

The quote-to-cash visual shows CRM, the orchestration state stores, the product/SKU system, the pricing/finance/legal system, the transient agent waves with deallocation, and the final quote package. The generic pattern diagram (via `display_scenario_flow`) shows how this pattern coordinates the agents.

In [ ]:
from release_room.diagram_helpers import display_quote_to_cash_flow, display_scenario_flow, quote_to_cash_flow_diagram

q2c = quote_to_cash_flow_diagram(scenario)
print(q2c.mermaid)
display_quote_to_cash_flow(scenario)
flow = display_scenario_flow(scenario)
flow.mermaid

## 7. MCP Tool Context

The agents are grounded by the local `quote_to_cash_context` MCP server (FastMCP over stdio): no network, no credentials, no writes. Below we list the tools it exposes and make one deterministic sample call.

In [ ]:
from release_room.mcp_servers import quote_to_cash_context

context = mcp_tool_context(scenario)
print("Server exposes:", quote_to_cash_context.AVAILABLE_TOOLS)
print("Tools used in this scenario:", context["all_tools_used"])
# One deterministic sample call into the same functions the stdio server exposes.
quote_to_cash_context.crm_get_quote_trigger("OPP-5001")

## 8. Agent Roster And Tool Access Map

The roster lists the agents; the tool access map shows which MCP tools each agent may call.

In [ ]:
roster = agent_roster(scenario)
tool_access_map = context["tools_by_agent"]
{"roster": roster, "tool_access_map": tool_access_map}

## 9. Orchestration State Handoff

The quote-to-cash flow is built around **stored context and transient agent waves**:

1. **Wave 1 (CRM):** `QuoteTriggerAgent` confirms the trigger conditions and `CustomerContextAgent` enriches the customer profile. The orchestration stores the **customer context** and deallocates wave 1.
2. **Wave 2 (Product/SKU):** `SkuDiscoveryAgent` and `ProductFitAgent` work against the product system. The orchestration stores the **product context** and deallocates wave 2.
3. **Wave 3 (Pricing/Legal + Quote):** `PricingTermsAgent` resolves pricing and terms, then `QuoteGenerationAgent` produces the final package.

Each pattern realizes this differently: Sequential passes state straight down the chain; Concurrent aggregates parallel findings; Handoff routes dynamically; Group Chat debates before storing conclusions; Magentic lets a manager re-plan which wave to run next. The Mermaid diagram above shows the stores and the deallocation edges.

## 10. Workflow Construction

This uses the same builder path as the `/responses` server, but runs in-process.

In [ ]:
from release_room.agents import build_ollama_config
from release_room.workflows import build_release_workflow, default_sample_max_tokens

config = build_ollama_config(**default_ollama_kwargs())
max_tokens = default_sample_max_tokens(scenario)  # 500 for Magentic, 250 otherwise
workflow = build_release_workflow(
    "scenario-16-quote-to-cash-handoff",
    model=config.model,
    ollama_host=config.host,
    temperature=config.temperature,
    num_ctx=config.num_ctx,
    max_tokens=max_tokens,
    keep_alive=config.keep_alive,
    think=config.think,
)
workflow

## 11. Live In-Process Run

This calls Ollama and launches the local MCP stdio server automatically (no approval prompts).

In [ ]:
result = await workflow.run(scenario.sample_input)
print(workflow_result_to_text(result)[:5000])

## 12. API Boundary Reference

Compare the in-process workflow with the hosted Responses endpoint. The MCP tools stay server-side.

In [ ]:
responses_api_reference(scenario)

## 13. Experiments

Try a different opportunity id (e.g. `OPP-5002`, which is not quote-ready) in the MCP tool cell, change a discount in `pricing_calculate_quote`, adjust `max_tokens`, or edit an agent instruction in the scenario module. Then compare this pattern against the other four Scenario 16 variants to see how coordination changes while the business story stays the same.